In [56]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

In [57]:
spark = SparkSession.builder \
    .appName("Stock Big Data ETL") \
    .getOrCreate()

spark

In [58]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/home/ridhi/projects/finance_project/data/processed/master.csv")

In [59]:
start = time.time()

print("Rows:", df.count())
print("Columns:", len(df.columns))

end = time.time()

print(f"Time Taken: {end-start:.2f} seconds")

Rows: 4668
Columns: 103
Time Taken: 0.99 seconds


In [60]:
df.printSchema()

root
 |-- Name_bal: string (nullable = true)
 |-- BSE Code_bal: double (nullable = true)
 |-- NSE Code_bal: string (nullable = true)
 |-- Industry_bal: string (nullable = true)
 |-- Current Price_bal: double (nullable = true)
 |-- Debt: double (nullable = true)
 |-- Equity capital: double (nullable = true)
 |-- Preference capital: double (nullable = true)
 |-- Reserves: double (nullable = true)
 |-- Secured loan: double (nullable = true)
 |-- Unsecured loan: double (nullable = true)
 |-- Balance sheet total: double (nullable = true)
 |-- Gross block: double (nullable = true)
 |-- Revaluation reserve: double (nullable = true)
 |-- Accumulated depreciation: double (nullable = true)
 |-- Net block: double (nullable = true)
 |-- Capital work in progress: double (nullable = true)
 |-- Investments: double (nullable = true)
 |-- Current assets: double (nullable = true)
 |-- Current liabilities: double (nullable = true)
 |-- Book value of unquoted investments: double (nullable = true)
 |-- Mar

In [61]:
df.show(5, truncate=False)

+----------------+------------+------------+-------------------------------------+-----------------+-------+--------------+------------------+--------+------------+--------------+-------------------+-----------+-------------------+------------------------+---------+------------------------+-----------+--------------+-------------------+----------------------------------+----------------------------------+----------------------+------------+---------------+-----------------+---------+-----------------+----------+----------------+----------------------+--------------+--------------------------------------+-------------------+------------------------------+------------------------+--------------------------+---------------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+----------------+----------------+----------------+-----------------+---------------------+---------------------+--------------------

In [62]:
df.describe().show()

+-------+---------------+------------------+------------+--------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+-------------------+-----------------+-------------------+------------------------+------------------+------------------------+------------------+-----------------+-------------------+----------------------------------+----------------------------------+----------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+----------------------+-----------------+--------------------------------------+-------------------+------------------------------+------------------------+--------------------------+---------------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+-----------------+-----------------+-------------

In [63]:
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.show(vertical=True, truncate=False)

-RECORD 0-----------------------------------------
 Name_bal                                  | 0    
 BSE Code_bal                              | 0    
 NSE Code_bal                              | 0    
 Industry_bal                              | 0    
 Current Price_bal                         | 1    
 Debt                                      | 16   
 Equity capital                            | 36   
 Preference capital                        | 109  
 Reserves                                  | 16   
 Secured loan                              | 109  
 Unsecured loan                            | 109  
 Balance sheet total                       | 16   
 Gross block                               | 109  
 Revaluation reserve                       | 109  
 Accumulated depreciation                  | 109  
 Net block                                 | 16   
 Capital work in progress                  | 16   
 Investments                               | 16   
 Current assets                

In [64]:
from pyspark.sql.functions import isnan

missing = df.select([
    (
        count(
            when(
                col(c).isNull() | isnan(c),
                c
            )
        ).alias(c)
    )
    if dict(df.dtypes)[c] in ["double", "float", "int", "bigint"]
    else count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

missing.show(vertical=True, truncate=False)

-RECORD 0-----------------------------------------
 Name_bal                                  | 0    
 BSE Code_bal                              | 0    
 NSE Code_bal                              | 0    
 Industry_bal                              | 0    
 Current Price_bal                         | 1    
 Debt                                      | 16   
 Equity capital                            | 36   
 Preference capital                        | 109  
 Reserves                                  | 16   
 Secured loan                              | 109  
 Unsecured loan                            | 109  
 Balance sheet total                       | 16   
 Gross block                               | 109  
 Revaluation reserve                       | 109  
 Accumulated depreciation                  | 109  
 Net block                                 | 16   
 Capital work in progress                  | 16   
 Investments                               | 16   
 Current assets                

In [65]:
drop_cols = [
    "Name_rat",
    "NSE Code_rat",
    "Industry_rat",
    "Days Receivable Outstanding",
    "signal",
    "t_1_price"
]

existing = [c for c in drop_cols if c in df.columns]

df = df.drop(*existing)

print(len(df.columns))

97


In [66]:
df = df.filter(col("future_return").isNotNull())

print(df.count())

4667


In [67]:
from pyspark.sql.types import StringType, NumericType

cat_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

num_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print("Categorical:", cat_cols)
print("Number of numerical columns:", len(num_cols))

Categorical: ['Name_bal', 'NSE Code_bal', 'Industry_bal', 'join_key', 'Credit rating']
Number of numerical columns: 92


In [68]:
df = df.drop("join_key")

In [69]:
df = df.drop(
    "Name_bal",
    "NSE Code_bal"
)

In [70]:
df = (
    df.withColumnRenamed("Industry_bal", "Industry")
      .withColumnRenamed("Current Price_bal", "Current Price")
      .withColumnRenamed("Market Capitalization_bal", "Market Capitalization")
)

In [71]:
from pyspark.sql.types import StringType

[
    f.name
    for f in df.schema.fields
    if isinstance(f.dataType, StringType)
]

['Industry', 'Credit rating']

In [72]:
missing = df.select([
    (
        count(when(col(c).isNull(), c)) / df.count()
    ).alias(c)
    for c in df.columns
])

missing.show(truncate=False)

+------------+--------+-------------+---------------------+-------------------+--------------------+---------------------+--------------------+--------------------+---------------------+--------------------+--------------------+------------------------+---------------------+------------------------+---------------------+---------------------+---------------------+----------------------------------+----------------------------------+----------------------+---------------------+---------------------+---------------------+---------------------+---------------------+--------------------+---------------------+----------------------+---------------------+--------------------------------------+--------------------+------------------------------+------------------------+--------------------------+---------------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+------------------+-------------------+--------

In [73]:
missing_long = (
    missing
    .selectExpr(
        "stack({}, {}) as (Column, MissingPercent)".format(
            len(df.columns),
            ",".join([f"'{c}', `{c}`" for c in df.columns])
        )
    )
)

missing_long.orderBy(
    col("MissingPercent").desc()
).show(20, truncate=False)

+-------------------------------+-------------------+
|Column                         |MissingPercent     |
+-------------------------------+-------------------+
|Credit rating                  |1.0                |
|Days Payable Outstanding       |0.35718877223055495|
|Days Inventory Outstanding     |0.30490679237197343|
|Working capital 10Years back   |0.28605099635740305|
|Debt 10Years back              |0.28605099635740305|
|Graham Number                  |0.2620527105206771 |
|Inventory turnover ratio       |0.2500535676023141 |
|Price to Earning               |0.22305549603599742|
|Debt 7Years back               |0.2022712663381187 |
|Net block 7Years back          |0.2022712663381187 |
|Working capital 7Years back    |0.2022712663381187 |
|Net block 5Years back          |0.12427683736875937|
|Debt 5Years back               |0.12427683736875937|
|Working capital 5Years back    |0.12427683736875937|
|Return on equity preceding year|0.11099207199485751|
|Price to book value        

In [74]:

df = df.drop("Credit rating")

In [75]:
"Credit rating" in df.columns

False

In [76]:
from pyspark.sql.types import NumericType

num_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(len(num_cols))

92


In [77]:

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in num_cols
]).show()

+------------+-------------+----+--------------+------------------+--------+------------+--------------+-------------------+-----------+-------------------+------------------------+---------+------------------------+-----------+--------------+-------------------+----------------------------------+----------------------------------+----------------------+------------+---------------+-----------------+---------+-----------------+----------+----------------+----------------------+--------------+--------------------------------------+-------------------+------------------------------+------------------------+--------------------------+---------------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+----------------+----------------+----------------+-----------------+---------------------+---------------------+---------------------+---------------------+------------+-----------------+--------------------

In [78]:
[c for c in df.columns if c.endswith("_bal")]

['BSE Code_bal']

In [79]:
[c for c in df.columns if c.endswith("_rat")]

['BSE Code_rat', 'Current Price_rat', 'Market Capitalization_rat']

In [80]:
[c for c in df.columns if "Price" in c]

['Current Price',
 'Current Price_rat',
 'Price to Earning',
 'Price to book value']

In [81]:
[c for c in df.columns if "Market" in c]

['Market value of quoted investments',
 'Market Capitalization',
 'Market Capitalization_rat']

In [82]:
df = df.drop(
    "BSE Code_bal",
    "BSE Code_rat",
    "Current Price_rat",
    "Market Capitalization_rat"
)

In [83]:
[c for c in df.columns if c.endswith("_bal")]

[]

In [84]:
[c for c in df.columns if c.endswith("_rat")]

[]

In [85]:
[c for c in df.columns if "future" in c.lower()]

['future_return']

In [86]:
[c for c in df.columns if "signal" in c.lower()]

[]

In [87]:
[c for c in df.columns if "target" in c.lower()]

[]

In [88]:
[c for c in df.columns if "return" in c.lower()]

['Return on assets',
 'Return on equity',
 'Return on invested capital',
 'Return on capital employed preceding year',
 'Return on assets preceding year',
 'Return on equity preceding year',
 'future_return']

In [89]:
[c for c in df.columns if "price" in c.lower()]

['Current Price', 'Price to Earning', 'Price to book value']

In [90]:
target = "future_return"

feature_cols = [
    c for c in df.columns
    if c != target
]

print(len(feature_cols))
print(target in feature_cols)

88
False


In [91]:
print(df.columns)

['Industry', 'Current Price', 'Debt', 'Equity capital', 'Preference capital', 'Reserves', 'Secured loan', 'Unsecured loan', 'Balance sheet total', 'Gross block', 'Revaluation reserve', 'Accumulated depreciation', 'Net block', 'Capital work in progress', 'Investments', 'Current assets', 'Current liabilities', 'Book value of unquoted investments', 'Market value of quoted investments', 'Contingent liabilities', 'Total Assets', 'Working capital', 'Lease liabilities', 'Inventory', 'Trade receivables', 'Face value', 'Cash Equivalents', 'Advance from Customers', 'Trade Payables', 'Number of equity shares preceding year', 'Debt preceding year', 'Working capital preceding year', 'Net block preceding year', 'Gross block preceding year', 'Capital work in progress preceding year', 'Working capital 3Years back', 'Working capital 5Years back', 'Working capital 7Years back', 'Working capital 10Years back', 'Debt 3Years back', 'Debt 5Years back', 'Debt 7Years back', 'Debt 10Years back', 'Net block 3Ye

In [92]:
len(df.columns)

89

In [94]:
num_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, NumericType)
]

print(len(num_cols))

88


In [95]:
cat_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

print(len(cat_cols))

1


In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(
    inputCols=num_cols,
    outputCols=num_cols,
    strategy="median"
)

imputer_model = imputer.fit(df)
imputer_model.write().overwrite().save(
    "../models/imputer_model"
)


imputed_df = imputer_model.transform(df)

In [ ]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="Industry",
    outputCol="Industry_Index",
    handleInvalid="keep"
)

indexer_model = indexer.fit(imputed_df)
indexer_model.write().overwrite().save(
    "../models/indexer_model"
)

indexed_df = indexer_model.transform(imputed_df)

In [98]:
df.select(
    "Industry",
    "Industry_Index"
).show(10, truncate=False)

+-----------------------------------------+--------------+
|Industry                                 |Industry_Index|
+-----------------------------------------+--------------+
|Mining / Minerals / Metals               |18.0          |
|Finance & Investments                    |0.0           |
|Finance & Investments                    |0.0           |
|Healthcare                               |19.0          |
|Computers - Software - Medium / Small    |4.0           |
|Computers - Software - Medium / Small    |4.0           |
|Diversified - Large                      |89.0          |
|Finance & Investments                    |0.0           |
|Entertainment / Electronic Media Software|11.0          |
|Finance & Investments                    |0.0           |
+-----------------------------------------+--------------+
only showing top 10 rows



In [99]:
from pyspark.ml.feature import OneHotEncoder

In [ ]:
encoder = OneHotEncoder(
    inputCols=["Industry_Index"],
    outputCols=["Industry_OHE"]
)

encoder_model = encoder.fit(indexed_df)
encoder_model.write().overwrite().save(
    "../models/encoder_model"
)

encoded_df = encoder_model.transform(indexed_df)

In [ ]:
encoded_df.select(
    "Industry",
    "Industry_Index",
    "Industry_OHE"
).show(10, truncate=False)

+-----------------------------------------+--------------+----------------+
|Industry                                 |Industry_Index|Industry_OHE    |
+-----------------------------------------+--------------+----------------+
|Mining / Minerals / Metals               |18.0          |(107,[18],[1.0])|
|Finance & Investments                    |0.0           |(107,[0],[1.0]) |
|Finance & Investments                    |0.0           |(107,[0],[1.0]) |
|Healthcare                               |19.0          |(107,[19],[1.0])|
|Computers - Software - Medium / Small    |4.0           |(107,[4],[1.0]) |
|Computers - Software - Medium / Small    |4.0           |(107,[4],[1.0]) |
|Diversified - Large                      |89.0          |(107,[89],[1.0])|
|Finance & Investments                    |0.0           |(107,[0],[1.0]) |
|Entertainment / Electronic Media Software|11.0          |(107,[11],[1.0])|
|Finance & Investments                    |0.0           |(107,[0],[1.0]) |
+-----------

In [ ]:
encoded_df = encoded_df.drop(
    "Industry",
    "Industry_Index"
)

In [ ]:
feature_cols = [
    c
    for c in encoded_df.columns
    if c not in ["future_return"]
]

print(feature_cols)

['Current Price', 'Debt', 'Equity capital', 'Preference capital', 'Reserves', 'Secured loan', 'Unsecured loan', 'Balance sheet total', 'Gross block', 'Revaluation reserve', 'Accumulated depreciation', 'Net block', 'Capital work in progress', 'Investments', 'Current assets', 'Current liabilities', 'Book value of unquoted investments', 'Market value of quoted investments', 'Contingent liabilities', 'Total Assets', 'Working capital', 'Lease liabilities', 'Inventory', 'Trade receivables', 'Face value', 'Cash Equivalents', 'Advance from Customers', 'Trade Payables', 'Number of equity shares preceding year', 'Debt preceding year', 'Working capital preceding year', 'Net block preceding year', 'Gross block preceding year', 'Capital work in progress preceding year', 'Working capital 3Years back', 'Working capital 5Years back', 'Working capital 7Years back', 'Working capital 10Years back', 'Debt 3Years back', 'Debt 5Years back', 'Debt 7Years back', 'Debt 10Years back', 'Net block 3Years back', '

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
vector_model = assembler.fit(encoded_df)
vector_df = assembler.transform(encoded_df)

vector_model.write().overwrite().save(
    "../models/vector_model"
)

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

In [ ]:
scaler_model = scaler.fit(vector_df)

scaler_model.write().overwrite().save(
    "../models/scaler_model"
)
df= scaler.transform(vector_df)

In [105]:
df.select(
    "features",
    "future_return"
).show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+
|features                                                                                                                                                                                                                        

In [107]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

train_df, test_df = df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print(train_df.count(), test_df.count())

3778 889


In [108]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="future_return"
)

lr_model = lr.fit(train_df)

26/06/25 17:22:02 WARN Instrumentation: [9cd9af7f] regParam is zero, which might cause numerical instability and overfitting.
26/06/25 17:22:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/25 17:22:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/06/25 17:22:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/06/25 17:22:02 WARN Instrumentation: [9cd9af7f] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


In [109]:
lr_pred = lr_model.transform(test_df)

lr_pred.select(
    "future_return",
    "prediction"
).show(5)

+--------------------+--------------------+
|       future_return|          prediction|
+--------------------+--------------------+
|-0.06060606060606...|-0.02726496025672...|
|-0.04347826086956525|-0.03436451417482355|
|-0.14285714285714288|-0.03993735837431631|
|                 0.0|-0.03545618943500307|
|0.030303030303030328|-0.03747718251184776|
+--------------------+--------------------+
only showing top 5 rows



In [110]:
rmse_eval = RegressionEvaluator(
    labelCol="future_return",
    predictionCol="prediction",
    metricName="rmse"
)

r2_eval = RegressionEvaluator(
    labelCol="future_return",
    predictionCol="prediction",
    metricName="r2"
)

print("RMSE :", rmse_eval.evaluate(lr_pred))
print("R²   :", r2_eval.evaluate(lr_pred))

RMSE : 0.14497650713151608
R²   : -3.072471794519106


In [111]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="future_return",
    seed=42
)

dt_model = dt.fit(train_df)

dt_pred = dt_model.transform(test_df)

print("Decision Tree RMSE:",
      rmse_eval.evaluate(dt_pred))

print("Decision Tree R²:",
      r2_eval.evaluate(dt_pred))

Decision Tree RMSE: 0.061211210732283604
Decision Tree R²: 0.27401982675253267


In [112]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="future_return",
    numTrees=100,
    seed=42
)

rf_model = rf.fit(train_df)

rf_pred = rf_model.transform(test_df)

print("Random Forest RMSE:",
      rmse_eval.evaluate(rf_pred))

print("Random Forest R²:",
      r2_eval.evaluate(rf_pred))

26/06/25 17:22:20 WARN DAGScheduler: Broadcasting large task binary with size 1202.4 KiB


Random Forest RMSE: 0.05813637178885889
Random Forest R²: 0.34512461050380405


In [113]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="future_return",
    numTrees=300,
    maxDepth=12,
    maxBins=64,
    minInstancesPerNode=3,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_df)

rf_pred = rf_model.transform(test_df)

print("RMSE:", rmse_eval.evaluate(rf_pred))
print("R²:", r2_eval.evaluate(rf_pred))

26/06/25 17:22:29 WARN DAGScheduler: Broadcasting large task binary with size 1151.2 KiB
26/06/25 17:22:32 WARN DAGScheduler: Broadcasting large task binary with size 1969.0 KiB
26/06/25 17:22:36 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/06/25 17:22:41 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB
26/06/25 17:22:45 WARN DAGScheduler: Broadcasting large task binary with size 1556.1 KiB
26/06/25 17:22:50 WARN DAGScheduler: Broadcasting large task binary with size 10.5 MiB
26/06/25 17:22:54 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/25 17:23:04 WARN DAGScheduler: Broadcasting large task binary with size 12.4 MiB
26/06/25 17:23:12 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 17:23:22 WARN DAGScheduler: Broadcasting large task binary with size 14.1 MiB
26/06/25 17:23:27 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/25 17:23:47 WARN DAGScheduler: Broad

RMSE: 0.056466778559598176


R²: 0.3821986622444684


In [120]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="future_return",
    maxIter=100,
    seed=42
)

gbt_model = gbt.fit(train_df)

gbt_pred = gbt_model.transform(test_df)

print("GBT RMSE:",
      rmse_eval.evaluate(gbt_pred))

print("GBT R²:",
      r2_eval.evaluate(gbt_pred))

GBT RMSE: 0.060286438831350736
GBT R²: 0.29579017229561566


In [115]:
rf_model.write().overwrite().save("../models/spark_rf_model")


26/06/25 17:25:36 WARN TaskSetManager: Stage 855 contains a task of very large size (3447 KiB). The maximum recommended task size is 1000 KiB.


In [116]:
from pyspark.ml.regression import RandomForestRegressionModel

loaded_model = RandomForestRegressionModel.load("../models/spark_rf_model")

print(type(loaded_model))

<class 'pyspark.ml.regression.RandomForestRegressionModel'>
